# ChuckleNet: Hybrid Extraction (Existing WavLM + New Prosody)

**Strategy:**
- Use existing 768-dim WavLM embeddings from `wavlm_utterance_safe` (already extracted!)
- Extract 21-dim prosody from existing WAV files (fast - just librosa)
- Match labels from `utterances_clean.jsonl`

**Time:** ~30-60 min vs 30+ hours for full extraction

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

In [ ]:
# Step 2: Load existing WavLM embeddings from wavlm_utterance_safe
import json
import subprocess
from pathlib import Path

WAVLM_DIR = Path('/content/gdrive/MyDrive/wavlm_utterance_safe')

print('📂 Loading existing WavLM embeddings...')
wavlm_data = {}  # video_id -> list of {start, end, embedding}

for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    wavlm_data[vid] = data['embeddings']

print(f'✅ Loaded WavLM for {len(wavlm_data)} videos')

# Show stats
total_utterances = sum(len(v) for v in wavlm_data.values())
print(f'   Total utterances with embeddings: {total_utterances}')

In [ ]:
# Step 3: Load utterances to get labels and match to embeddings
UTT_PATH = Path('/content/gdrive/MyDrive/utterances_clean.jsonl')
if not UTT_PATH.exists():
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True, capture_output=True)
    subprocess.run(['gdown', '--fuzzy', '-O', str(UTT_PATH),
        'https://drive.google.com/file/d/1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn/view'],
        check=True, capture_output=True)

print('📋 Loading utterances with labels...')
utterances = []
with open(UTT_PATH) as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'✅ Loaded {len(utterances)} utterances')

# Create lookup: (video_id, start, end) -> label
label_lookup = {}
for u in utterances:
    key = (u['video_id'], round(u['start'], 2), round(u['end'], 2))
    label_lookup[key] = u['label']

print(f'   Labeled utterances: {len(label_lookup)}')

In [ ]:
# Step 4: Match WavLM embeddings with labels
matched = 0
unmatched = 0

for vid, embeddings in wavlm_data.items():
    for emb in embeddings:
        key = (vid, round(emb['start'], 2), round(emb['end'], 2))
        if key in label_lookup:
            emb['label'] = label_lookup[key]
            matched += 1
        else:
            emb['label'] = 0  # default to no laughter
            unmatched += 1

print(f'✅ Matched: {matched} utterances with labels')
print(f'⚠️ Unmatched (defaulted to 0): {unmatched}')

# Count positive/negative
pos_count = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 1)
neg_count = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 0)
print(f'   Positive: {pos_count} ({pos_count/(pos_count+neg_count)*100:.1f}%)')
print(f'   Negative: {neg_count} ({neg_count/(pos_count+neg_count)*100:.1f}%)')

In [ ]:
# Step 5: Split into train/val/test (by video for no leakage)
import random

video_ids = list(wavlm_data.keys())
random.seed(42)
random.shuffle(video_ids)

n = len(video_ids)
val_vids = set(video_ids[:int(n*0.1)])
test_vids = set(video_ids[int(n*0.1):int(n*0.2)])

train_data = []
val_data = []
test_data = []

for vid in video_ids:
    if vid in val_vids:
        val_data.extend(wavlm_data[vid])
    elif vid in test_vids:
        test_data.extend(wavlm_data[vid])
    else:
        train_data.extend(wavlm_data[vid])

print(f'📊 Split:')
print(f'   Train: {len(train_data)} utterances ({len(set(d["video_id"] for d in train_data))} videos)')
print(f'   Val: {len(val_data)} utterances ({len(set(d["video_id"] for d in val_data))} videos)')
print(f'   Test: {len(test_data)} utterances ({len(set(d["video_id"] for d in test_data))} videos)')

## Step 6: Extract Prosody (21-dim) for existing WAV files

This is MUCH faster than WavLM extraction - just librosa feature extraction.

In [ ]:
# Step 6a: Find existing WAV files or audio files
from pathlib import Path

BASE = Path('/content/gdrive/MyDrive')

# Try different audio sources
audio_files = {}  # video_id -> path

# 1. Check for WAV in audio_final (these are already WAV)
audio_final = BASE / 'chuckle_audio_all' / 'audio_final'
if audio_final.exists():
    for p in audio_final.glob('*.wav'):
        if not p.name.endswith('.part'):
            audio_files[p.stem] = str(p)

print(f'🎵 Found {len(audio_files)} WAV files in audio_final')

# 2. If not enough, check audio_new
if len(audio_files) < len(wavlm_data):
    audio_new = BASE / 'chuckle_audio_all' / 'audio_new'
    if audio_new.exists():
        for p in audio_new.glob('*.wav'):
            if not p.name.endswith('.part'):
                audio_files[p.stem] = str(p)
    print(f'   After audio_new: {len(audio_files)} total')

# 3. Check chuckle_audio (MP3 files)
if len(audio_files) < len(wavlm_data):
    chuckle_audio = BASE / 'chuckle_audio'
    if chuckle_audio.exists():
        for p in chuckle_audio.glob('*.mp3'):
            audio_files[p.stem] = str(p)
    print(f'   After chuckle_audio: {len(audio_files)} total')

print(f'📁 Audio files available: {len(audio_files)}')
print(f'📁 WavLM embeddings available: {len(wavlm_data)}')
print(f'📁 Coverage: {len(audio_files & set(wavlm_data.keys()))} videos')

In [ ]:
# Step 6b: Extract prosody for each video
import librosa
import numpy as np
import soundfile as sf
import time

SR = 16000

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    features.extend([
        len(y) / sr,
        len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)
    ])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat),
        np.mean(zcr), np.std(zcr)
    ])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

print('🔍 Extracting prosody for each video...')
t0 = time.time()

# Load checkpoint
PROSODY_CKPT = BASE / 'prosody_checkpoint.json'
if PROSODY_CKPT.exists():
    with open(PROSODY_CKPT) as f:
        prosody_data = json.load(f)
    done_vids = set(prosody_data.get('done_videos', []))
    print(f'📂 Resume: {len(done_vids)} already done')
else:
    prosody_data = {}
    done_vids = set()

total_videos = len(wavlm_data)

for i, vid in enumerate(wavlm_data.keys()):
    if vid in done_vids:
        continue
    
    audio_path = audio_files.get(vid)
    if not audio_path:
        continue
    
    try:
        # Load audio
        if audio_path.endswith('.wav'):
            y, sr = sf.read(audio_path, dtype='float32')
        else:
            y, sr = librosa.load(audio_path, sr=SR, mono=True)
        
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        if sr != SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        
        # Store for slicing
        prosody_data[vid] = {'audio': y, 'sr': SR}
        done_vids.add(vid)
        
    except Exception as e:
        continue
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (total_videos - i - 1)
        print(f'📊 {i+1}/{total_videos} | ETA: {eta/60:.1f} min')
        
        # Save checkpoint
        with open(PROSODY_CKPT, 'w') as f:
            json.dump({'done_videos': list(done_vids)}, f)

print(f'\n✅ Loaded audio for {len(prosody_data)} videos in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Step 6c: Extract prosody per utterance
print('🔍 Extracting prosody per utterance...')
t0 = time.time()

prosody_ckpt = BASE / 'prosody_per_utterance.json'
if prosody_ckpt.exists():
    with open(prosody_ckpt) as f:
        all_prosody = json.load(f)
else:
    all_prosody = {}

done_count = len(all_prosody)
print(f'📂 Resume: {done_count} already done')

for i, vid in enumerate(wavlm_data.keys()):
    if vid in all_prosody:
        continue
    
    if vid not in prosody_data:
        continue
    
    y = prosody_data[vid]['audio']
    sr = prosody_data[vid]['sr']
    
    video_prosody = []
    for emb in wavlm_data[vid]:
        start_s = emb['start']
        end_s = emb['end']
        
        start_sample = int(start_s * sr)
        end_sample = int(end_s * sr)
        
        if end_sample > len(y):
            end_sample = len(y)
        
        y_slice = y[start_sample:end_sample]
        
        if len(y_slice) < sr * 0.1:
            video_prosody.append(np.zeros(21, dtype=np.float32).tolist())
        else:
            prosody = extract_prosody_21dim(y_slice, sr)
            video_prosody.append(prosody.tolist())
    
    all_prosody[vid] = video_prosody
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'📊 {i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f} min')
        
        with open(prosody_ckpt, 'w') as f:
            json.dump(all_prosody, f)

print(f'\n✅ Extracted prosody for {len(all_prosody)} videos in {(time.time()-t0)/60:.1f} min')

## Step 7: Combine WavLM + Prosody and Train

In [ ]:
# Step 7: Combine features and prepare for training
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

class FusionDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        wavlm = torch.tensor(d['wavlm'], dtype=torch.float32)
        prosody = torch.tensor(d['prosody'], dtype=torch.float32)
        x = torch.cat([wavlm, prosody])  # 768 + 21 = 789
        y = torch.tensor(d['label'], dtype=torch.float32)
        return x, y

# Combine train/val/test
def prepare_data(data_list, prosody_dict):
    result = []
    for d in data_list:
        vid = d['video_id']
        if vid not in prosody_dict:
            continue
        
        # Find index of this utterance
        for emb in wavlm_data[vid]:
            if abs(emb['start'] - d['start']) < 0.01 and abs(emb['end'] - d['end']) < 0.01:
                idx = wavlm_data[vid].index(emb)
                if idx < len(prosody_dict[vid]):
                    result.append({
                        'wavlm': emb['embedding'],
                        'prosody': prosody_dict[vid][idx],
                        'label': d['label']
                    })
                break
    return result

print('📦 Preparing datasets...')
train_ds_data = prepare_data(train_data, all_prosody)
val_ds_data = prepare_data(val_data, all_prosody)
test_ds_data = prepare_data(test_data, all_prosody)

train_ds = FusionDataset(train_ds_data)
val_ds = FusionDataset(val_ds_data)
test_ds = FusionDataset(test_ds_data)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Step 8: Train Fusion Model
import torch.nn as nn

class FusionModel(nn.Module):
    def __init__(self, input_dim=789):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

model = FusionModel().to(device)

# Class weights
all_labels = [d['label'] for d in train_ds_data]
pos_weight = compute_class_weight('balanced', classes=[0,1], y=all_labels)[1]
pos_weight = torch.tensor([pos_weight]).to(device)
print(f'⚖️ Pos weight: {pos_weight.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            val_preds.extend((out > 0.5).cpu().numpy())
            val_labels.extend(y.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, str(BASE / 'best_fusion_model.pt'))
        print(f'  ✅ New best!')

print(f'\n🏆 Best Val F1: {best_f1:.4f}')

In [ ]:
# Step 9: Final Evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        test_preds.extend((out > 0.5).cpu().numpy())
        test_labels.extend(y.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'🏆 Test F1: {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))